# Dual-Space Embedding and Alignment

This notebook builds and validates a **cross-space embedding** for comparing iGEM student projects with scientific papers.

## Why two spaces?

SPECTER (and SPECTER2) was trained on scientific paper abstracts and citation graphs.
iGEM project wikis are informal, interdisciplinary, and mix biology with hardware, ethics, and policy —
a very different register from the academic text the model was trained on.
Projecting both corpora through SPECTER would distort the iGEM side.

Instead:
- Embed **iGEM projects** with `all-MiniLM-L6-v2` — a general sentence encoder suited to informal text.
- Embed **scientific papers** with `allenai-specter` — trained on scientific titles and abstracts.
- Learn a **linear mapping** W from project space (384-d) → paper space (768-d) using known links.

## The anchor links

Both link types go through BioBrick parts:

| Type | Description | Source |
|------|-------------|--------|
| A — team cited paper | A BioBrick's documentation cites a paper | `part_source_papers.csv` |
| B — paper cited part | A paper mentions a BioBrick part by name | `paper_mentions_part.csv` |

## Alignment strategy

1. Build anchor pairs `(project_text, paper_text)` from both link types.
2. Embed each side with its respective model.
3. Fit a least-squares linear map W on the training anchors.
4. Evaluate: do linked pairs end up closer together than random pairs after mapping?
5. Optionally upgrade to a thin MLP adapter for non-linear alignment.
6. Run a geographic permutation test: does the aligned space show real city-level clustering?


In [ ]:
import re
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neural_network import MLPRegressor
from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
np.random.seed(42)
print('Libraries loaded.')

In [ ]:
DATA_DIR  = Path('../data/processed')
CACHE_DIR = Path('../data/embeddings')
FIG_DIR   = Path('../outputs/figures')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# The two embedding models
# all-MiniLM-L6-v2: 384-dimensional, fast, handles informal/conversational text well
# allenai-specter:  768-dimensional, trained on scientific titles + abstracts + citation graph
PROJECT_MODEL = 'all-MiniLM-L6-v2'
PAPER_MODEL   = 'allenai-specter'

VAL_FRACTION = 0.2   # fraction of anchor pairs held out for evaluation
RANDOM_SEED  = 42

In [ ]:
projects            = pd.read_csv(DATA_DIR / 'projects.csv')
papers              = pd.read_csv(DATA_DIR / 'papers.csv')
parts               = pd.read_csv(DATA_DIR / 'parts.csv')
part_source_papers  = pd.read_csv(DATA_DIR / 'part_source_papers.csv')   # Type A: team cited paper
paper_mentions_part = pd.read_csv(DATA_DIR / 'paper_mentions_part.csv')  # Type B: paper cited part
biobrick_papers     = pd.read_csv(DATA_DIR / 'biobrick_papers.csv')      # paper text for anchor papers

print(f'Projects:         {len(projects):,}')
print(f'Papers:           {len(papers):,}')
print(f'Parts:            {len(parts):,}')
print(f'Type-A links:     {len(part_source_papers):,}  (part cited paper)')
print(f'Type-B links:     {len(paper_mentions_part):,}  (paper cited part)')
print(f'BioBrick papers:  {len(biobrick_papers):,}  (full text for anchor papers)')

## Building anchor pairs

An anchor pair is a `(project, paper)` combination where we have evidence the two are semantically connected.
We build these through the BioBrick part graph.

### Project ID parsing
Project IDs follow the pattern `igem_{team_name}_{year}` (e.g. `igem_Aberdeen_Scotland_2009`).
The `parts.csv` file records which team created each part, also with a `team_name` field.
We match these by normalising team names to lowercase.

**Note:** Team name formats are not perfectly standardised across datasets.
The match rate below tells us what fraction of raw links survive to become usable anchor pairs.
Lost links are links where the team name format differed or where the paper DOI was not in our text index.


In [ ]:
# --- Paper text lookup: DOI → 'title [SEP] abstract' ---
# SPECTER was trained with title and abstract separated by [SEP].
# We use biobrick_papers.csv as the source because it has both fields cleanly separated.
doi_to_text = {}
for _, row in biobrick_papers.iterrows():
    if pd.notna(row.get('doi')) and pd.notna(row.get('abstract')) and pd.notna(row.get('title')):
        doi_to_text[row['doi']] = f"{row['title']} [SEP] {row['abstract']}"

print(f'Paper texts available by DOI: {len(doi_to_text):,}')

# --- Project lookup: (team_name_lower, year) → project_id ---
def parse_project_id(pid):
    """Extract (team_name, year) from igem_{team_name}_{year}."""
    m = re.match(r'^igem_(.+)_(\d{4})$', pid)
    if m:
        return m.group(1).lower(), int(m.group(2))
    return None, None

proj_lookup = {}  # (team_name_lower, year) -> project_id
for pid in projects['id']:
    team, year = parse_project_id(pid)
    if team:
        proj_lookup[(team, year)] = pid

print(f'Project IDs parsed: {len(proj_lookup):,}')

# --- Part → team lookup ---
# Some parts have no team recorded (team_id is NaN). We drop those.
part_to_team = (
    parts[['part_name', 'team_name', 'year']]
    .dropna(subset=['part_name', 'team_name'])
    .drop_duplicates('part_name')
)
print(f'Parts with known team: {len(part_to_team):,}  (of {len(parts):,} total)')

In [ ]:
# --- Type A: team cited paper ---
# A BioBrick's documentation cites a paper → that team is linked to that paper.
type_a = (
    part_source_papers
    .merge(part_to_team[['part_name', 'team_name', 'year']], on='part_name', how='inner')
    [['team_name', 'year', 'doi']]
    .dropna(subset=['doi'])
    .assign(link_type='team_cited_paper')
)

# --- Type B: paper cited part ---
# A published paper mentions a BioBrick by name → that paper is linked to the team who made the part.
type_b = (
    paper_mentions_part
    .merge(part_to_team[['part_name', 'team_name', 'year']], on='part_name', how='inner')
    [['team_name', 'year', 'doi']]
    .dropna(subset=['doi'])
    .assign(link_type='paper_cited_part')
)

raw_links = pd.concat([type_a, type_b]).drop_duplicates(['team_name', 'year', 'doi'])
print(f'Raw links: {len(raw_links):,}  (Type A: {len(type_a):,}, Type B: {len(type_b):,})')

In [ ]:
# Match each link to a project ID and to paper text
def find_project(team_name, year):
    """Look up a project ID by team name and year (case-insensitive)."""
    try:
        return proj_lookup.get((str(team_name).lower(), int(year)))
    except (ValueError, TypeError):
        return None

project_text_map = projects.set_index('id')['text'].to_dict()

raw_links = raw_links.copy()
raw_links['project_id']   = raw_links.apply(lambda r: find_project(r['team_name'], r['year']), axis=1)
raw_links['paper_text']   = raw_links['doi'].map(doi_to_text)
raw_links['project_text'] = raw_links['project_id'].map(project_text_map)

anchor_pairs = (
    raw_links
    .dropna(subset=['project_id', 'paper_text', 'project_text'])
    .drop_duplicates(['project_id', 'doi'])
    .reset_index(drop=True)
)

print(f'Usable anchor pairs:  {len(anchor_pairs):,}')
print(f'  Projects covered:   {anchor_pairs["project_id"].nunique():,}')
print(f'  Papers covered:     {anchor_pairs["doi"].nunique():,}')
print(f'  Link types:         {dict(anchor_pairs["link_type"].value_counts())}')
print()
n_lost_project = raw_links['project_id'].isna().sum()
n_lost_text    = raw_links['paper_text'].isna().sum()
print(f'Lost links: {n_lost_project:,} with no matching project ID, {n_lost_text:,} with no paper text')
print('  (Team name formats vary across datasets; unmatched links are dropped.)')

In [ ]:
# Inspect a few anchor pairs to verify the links make sense
proj_title_map  = projects.set_index('id')['title'].to_dict()
doi_to_title    = biobrick_papers.set_index('doi')['title'].to_dict()

sample = anchor_pairs.sample(min(5, len(anchor_pairs)), random_state=RANDOM_SEED)

for _, row in sample.iterrows():
    ptitle = proj_title_map.get(row['project_id'], '(no title)')[:80]
    rtitle = doi_to_title.get(row['doi'], '(no title)')[:80]
    print(f"Project [{row['link_type']}]: {ptitle}")
    print(f"Paper:                      {rtitle}")
    print()

## Embedding each corpus

**iGEM projects → `all-MiniLM-L6-v2` (384-dimensional)**  
A lightweight general-purpose sentence encoder that performs well on informal, varied text.
iGEM wikis are informal and interdisciplinary — a general model is more appropriate than a scientific one
that was never exposed to this kind of writing.

**Scientific papers → `allenai-specter` (768-dimensional)**  
SPECTER was trained on 146k scientific papers using titles, abstracts, and citation-graph supervision.
The intended input format is `"title [SEP] abstract"`. We use this for the anchor papers (from `biobrick_papers.csv`)
and for the full paper corpus (from `papers.csv`).

Embeddings are **normalised to unit length** before saving, so cosine similarity equals the dot product.
Embeddings are **cached to disk** — re-running this cell will load from the cache rather than re-encoding.


In [ ]:
def embed_with_cache(texts, model_name, cache_path, batch_size=64):
    """Embed a list of texts, writing/reading a pickle cache."""
    cache_path = Path(cache_path)
    if cache_path.exists():
        print(f'  Loading from cache: {cache_path.name}')
        with open(cache_path, 'rb') as f:
            return pickle.load(f)
    print(f'  Encoding {len(texts):,} texts with {model_name}...')
    model = SentenceTransformer(model_name)
    embs = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # unit vectors → cosine sim = dot product
    )
    with open(cache_path, 'wb') as f:
        pickle.dump(embs, f)
    print(f'  Cached to {cache_path.name}')
    return embs

In [ ]:
project_texts = projects['text'].fillna('').tolist()

print('Embedding iGEM projects:')
project_embs = embed_with_cache(
    project_texts,
    PROJECT_MODEL,
    CACHE_DIR / 'projects_miniLM.pkl',
)
print(f'  Shape: {project_embs.shape}  ({len(projects)} projects × {project_embs.shape[1]} dims)')

In [ ]:
# Embed the anchor papers (those we'll use to fit and evaluate the alignment).
# We use only the unique DOIs that appear in anchor_pairs.
unique_anchors = (
    anchor_pairs[['doi', 'paper_text']]
    .drop_duplicates('doi')
    .sort_values('doi')
    .reset_index(drop=True)
)
anchor_dois   = unique_anchors['doi'].tolist()
anchor_texts  = unique_anchors['paper_text'].tolist()

print('Embedding anchor papers:')
anchor_paper_embs = embed_with_cache(
    anchor_texts,
    PAPER_MODEL,
    CACHE_DIR / 'anchor_papers_specter.pkl',
)
doi_to_emb = dict(zip(anchor_dois, anchor_paper_embs))
print(f'  Shape: {anchor_paper_embs.shape}  ({len(anchor_dois)} unique papers × {anchor_paper_embs.shape[1]} dims)')

In [ ]:
# Embed the full paper corpus for downstream geographic analysis.
# papers.csv 'text' is title + ". " + abstract, which is close enough to SPECTER's expected format.
paper_texts_full = papers['text'].fillna('').tolist()

print('Embedding full paper corpus:')
paper_embs_full = embed_with_cache(
    paper_texts_full,
    PAPER_MODEL,
    CACHE_DIR / 'papers_specter.pkl',
)
print(f'  Shape: {paper_embs_full.shape}  ({len(papers)} papers × {paper_embs_full.shape[1]} dims)')

## Learning the alignment

We want a matrix W such that:

    project_embedding @ W  ≈  paper_embedding

Because project embeddings are 384-dimensional and paper embeddings are 768-dimensional,
W is a rectangular matrix (384 × 768) — not a square rotation.

We solve for W using **ordinary least squares (OLS)**, which minimises the sum of squared
distances between the mapped project embeddings and the target paper embeddings:

    W = (X^T X)^{-1} X^T Y

where X is the project embedding matrix for training anchors and Y is the paper embedding matrix.

We hold out 20% of anchor pairs for evaluation.
The key question: **after applying W, are linked pairs closer than random pairs?**


In [ ]:
# Build the (X, Y) matrices for all usable anchor pairs
proj_id_to_idx = {pid: i for i, pid in enumerate(projects['id'])}

X_rows, Y_rows = [], []
for _, row in anchor_pairs.iterrows():
    if row['project_id'] not in proj_id_to_idx:
        continue
    if row['doi'] not in doi_to_emb:
        continue
    X_rows.append(project_embs[proj_id_to_idx[row['project_id']]])
    Y_rows.append(doi_to_emb[row['doi']])

X_all = np.array(X_rows)   # (n_anchors, 384)
Y_all = np.array(Y_rows)   # (n_anchors, 768)

print(f'Anchor matrices: X={X_all.shape}, Y={Y_all.shape}')

if len(X_all) < 10:
    raise RuntimeError(
        f'Only {len(X_all)} anchor pairs — not enough to fit an alignment. '
        'Check that team name formats match between parts.csv and projects.csv.'
    )

# Train / validation split
X_train, X_val, Y_train, Y_val = train_test_split(
    X_all, Y_all,
    test_size=VAL_FRACTION,
    random_state=RANDOM_SEED,
)
print(f'Training anchors: {len(X_train)},  Validation anchors: {len(X_val)}')

# Fit the linear map with least squares
W, _, rank, _ = np.linalg.lstsq(X_train, Y_train, rcond=None)
print(f'W shape: {W.shape}  (maps {W.shape[0]}-d project space → {W.shape[1]}-d paper space)')
print(f'Matrix rank: {rank}')

In [ ]:
# Evaluate: are linked pairs closer after alignment than random pairs?

# Map validation project embeddings into paper space
X_val_mapped = X_val @ W   # (n_val, 768)

# Similarity between each mapped project and its actual linked paper
linked_sims = np.array([
    cosine_similarity(xm.reshape(1, -1), y.reshape(1, -1))[0, 0]
    for xm, y in zip(X_val_mapped, Y_val)
])

# Baseline: similarity between each mapped project and a random paper
rng = np.random.default_rng(RANDOM_SEED)
rand_idx  = rng.permutation(len(Y_val))
random_sims = np.array([
    cosine_similarity(xm.reshape(1, -1), Y_val[j].reshape(1, -1))[0, 0]
    for xm, j in zip(X_val_mapped, rand_idx)
])

# Mann-Whitney U: are linked similarities stochastically larger than random?
stat, p = stats.mannwhitneyu(linked_sims, random_sims, alternative='greater')

print(f'Linked pairs  — mean cosine similarity: {linked_sims.mean():.4f} (±{linked_sims.std():.4f})')
print(f'Random pairs  — mean cosine similarity: {random_sims.mean():.4f} (±{random_sims.std():.4f})')
print(f'Mann-Whitney U (linked > random):  U={stat:.0f},  p={p:.4f}')

if p < 0.05:
    print('✓ Linked pairs are significantly closer together than random pairs.')
else:
    print('✗ No significant difference detected — alignment may need more anchors.')

# Plot distributions
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(linked_sims, bins=25, alpha=0.7, label='Linked pairs',  color='steelblue')
ax.hist(random_sims, bins=25, alpha=0.7, label='Random pairs',  color='salmon')
ax.axvline(linked_sims.mean(), color='steelblue', linestyle='--', linewidth=1.5)
ax.axvline(random_sims.mean(), color='salmon',    linestyle='--', linewidth=1.5)
ax.set_xlabel('Cosine similarity after mapping to paper space')
ax.set_ylabel('Count')
ax.set_title(f'Alignment evaluation  (p = {p:.3f})')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'alignment_similarity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Optional: thin MLP adapter (non-linear alignment)
# Only worth training if we have enough anchor pairs.
# A single hidden layer with 512 units is enough — we're not training a deep network,
# just allowing a mild non-linearity beyond the linear map.

MIN_PAIRS_FOR_MLP = 200

if len(X_train) >= MIN_PAIRS_FOR_MLP:
    print(f'Training MLP adapter on {len(X_train)} pairs...')
    mlp = MLPRegressor(
        hidden_layer_sizes=(512,),
        activation='relu',
        max_iter=500,
        random_state=RANDOM_SEED,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20,
        verbose=False,
    )
    mlp.fit(X_train, Y_train)

    X_val_mlp = mlp.predict(X_val)
    mlp_sims = np.array([
        cosine_similarity(xm.reshape(1, -1), y.reshape(1, -1))[0, 0]
        for xm, y in zip(X_val_mlp, Y_val)
    ])
    print(f'MLP adapter — mean cosine similarity: {mlp_sims.mean():.4f}')
    print(f'Linear map  — mean cosine similarity: {linked_sims.mean():.4f}')
    use_mlp = mlp_sims.mean() > linked_sims.mean()
    print(f'Using: {"MLP adapter" if use_mlp else "linear map (MLP did not improve)"}')
else:
    use_mlp = False
    print(f'Only {len(X_train)} training pairs — below the {MIN_PAIRS_FOR_MLP} threshold for MLP.')
    print('Using the linear map. Collect more BioBrick links to unlock the MLP upgrade.')

# The function we'll use for projecting any project embedding into paper space
def project_to_paper_space(embs):
    """Map project embeddings (384-d) into paper space (768-d)."""
    if use_mlp:
        return mlp.predict(embs)
    return embs @ W

## Geographic coherence test

The core thesis claim is that projects and papers in the same city form a local semantic cluster.
After alignment we can test this directly:

1. Map all project embeddings into paper space.
2. Compute **intra-city cosine similarity**: the average similarity between artifacts from the same city.
3. Run a **permutation test**: shuffle city labels 500 times, recompute the score each time.
4. The p-value is the fraction of shuffles that beat the real score.

A significant result (p < 0.05) means cities cluster semantically — not just spatially.
This is the empirical signature of local innovation trajectories.

**What can make this fail:**
- Too many cities with only one artifact (no intra-city pairs to compute).
- City geocoding errors that assign artifacts to the wrong city.
- Not enough anchor pairs to produce a reliable alignment.


In [ ]:
# Map all project embeddings into paper space
all_proj_mapped = project_to_paper_space(project_embs)  # (n_projects, 768)

# Combine with full paper embeddings into one shared space
all_embs   = np.vstack([all_proj_mapped, paper_embs_full])   # (n_proj + n_papers, 768)
all_cities = (
    projects['city'].fillna('').tolist() +
    papers['city'].fillna('').tolist()
)

def intra_city_score(embs, cities):
    """Average cosine similarity of same-city pairs, across cities with >= 2 artifacts."""
    cities = np.array(cities)
    scores = []
    for city in set(cities):
        if not city:          # skip blanks
            continue
        idx = np.where(cities == city)[0]
        if len(idx) < 2:
            continue
        sims = cosine_similarity(embs[idx])
        upper = sims[np.triu_indices(len(idx), k=1)]   # upper triangle, no diagonal
        scores.append(upper.mean())
    return np.mean(scores) if scores else 0.0

print('Computing observed intra-city similarity...')
observed = intra_city_score(all_embs, all_cities)
print(f'Observed score: {observed:.4f}')

print('Running permutation test (500 shuffles)...')
rng = np.random.default_rng(RANDOM_SEED)
null_scores = [
    intra_city_score(all_embs, rng.permutation(all_cities))
    for _ in range(500)
]
null_scores = np.array(null_scores)
p_val = (null_scores >= observed).mean()

print(f'Null mean:      {null_scores.mean():.4f}  (±{null_scores.std():.4f})')
print(f'Observed:       {observed:.4f}')
print(f'p-value:        {p_val:.4f}')
if p_val < 0.05:
    print('✓ Cities cluster semantically — consistent with local innovation trajectories.')
else:
    print('✗ No significant geographic clustering detected.')

# Plot
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(null_scores, bins=40, color='lightgray', edgecolor='white', label='Shuffled cities')
ax.axvline(observed, color='steelblue', linewidth=2,
           label=f'Observed ({observed:.4f})')
ax.axvline(null_scores.mean(), color='gray', linestyle='--', linewidth=1.2,
           label=f'Null mean ({null_scores.mean():.4f})')
ax.set_xlabel('Mean intra-city cosine similarity')
ax.set_ylabel('Count (permutations)')
ax.set_title(f'Geographic coherence permutation test  (p = {p_val:.3f})')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'geographic_permutation_test.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save the alignment matrix and projected embeddings for downstream notebooks
np.save(CACHE_DIR / 'alignment_W.npy', W)
np.save(CACHE_DIR / 'projects_in_paper_space.npy', all_proj_mapped)

# Save combined metadata: one row per artifact, with embedding array index
combined_meta = []

for i, (_, row) in enumerate(projects.iterrows()):
    combined_meta.append({
        'id':              row['id'],
        'type':            'project',
        'title':           row.get('title', ''),
        'year':            int(row['year']) if pd.notna(row.get('year')) else None,
        'city':            row.get('city', ''),
        'country':         row.get('country', ''),
        'case_study_flag': bool(row.get('case_study_flag', False)),
        'proj_emb_index':  i,        # index into projects_in_paper_space.npy
        'embedding_source': 'projected_via_W',
    })

for i, (_, row) in enumerate(papers.iterrows()):
    combined_meta.append({
        'id':              row['id'],
        'type':            'paper',
        'title':           row.get('title', ''),
        'year':            int(row['year']) if pd.notna(row.get('year')) else None,
        'city':            row.get('city', ''),
        'country':         row.get('country', ''),
        'case_study_flag': bool(row.get('case_study_flag', False)),
        'paper_emb_index': i,        # index into papers_specter.pkl
        'embedding_source': 'specter',
    })

with open(CACHE_DIR / 'combined_meta.json', 'w') as f:
    json.dump(combined_meta, f, indent=2)

print('Saved to data/embeddings/:')
print(f'  alignment_W.npy              {W.shape}')
print(f'  projects_in_paper_space.npy  {all_proj_mapped.shape}')
print(f'  combined_meta.json           {len(combined_meta)} rows')

## Summary

This notebook implemented:

| Step | What it does |
|------|--------------|
| Anchor pairs | Link projects to papers via BioBrick part citations (two directions) |
| Project embedding | `all-MiniLM-L6-v2`, 384-d, suited to informal iGEM wiki text |
| Paper embedding | `allenai-specter`, 768-d, trained on scientific literature |
| Linear alignment | OLS least-squares map W (384 → 768) fitted on training anchors |
| Pair test | Mann-Whitney U: are linked pairs closer than random after mapping? |
| Geographic test | Permutation test: do same-city pairs cluster more than chance? |

### What limits the current alignment

- **Anchor count**: `part_source_papers.csv` has DOIs but `paper_id` is unfilled, so many Type-A links cannot be matched to paper text. Filling that column would increase the anchor set substantially.
- **Team name mismatch**: project IDs and parts team names use different formats. A crosswalk table would recover more links.
- **Single linear map**: W assumes the two spaces are globally linearly related. This is a reasonable starting point, but a more flexible model (MLP, locality-sensitive alignment) may help if the spaces are non-linearly related in different subfields.

### Next steps

1. **Cross-subfield evaluation**: train on all except carbon capture, test generalisation on carbon capture alone. This is the cleanest external validity check.
2. **Visualisation**: use `projects_in_paper_space.npy` and `papers_specter.pkl` as input to a shared UMAP projection in `01_city_level_analysis.ipynb`.
3. **Upgrade to SPECTER2**: if results look promising, swap `allenai-specter` for `allenai/specter2_proximity` (HuggingFace) for better scientific-text quality.
